In [ ]:
import pandas as pd
import numpy as np
import sys
import os

def process_ncbi_gene_info_integrated(target_organisms=None):
    """
    NCBI gene_info 데이터를 다운로드하여 여러 종의 데이터를 통합하고 
    3개의 정규화된 테이블로 분리하는 함수.
    
    Parameters:
    -----------
    target_organisms : dict
        {tax_id (int): organism_name (str)} 형태의 딕셔너리
        예: {9606: 'Homo sapiens', 10090: 'Mus musculus'}
        
    Returns:
    --------
    tuple: (df_main, df_synonyms, df_ensembl)
    """
    
    # 기본값 설정 (Human & Mouse)
    if target_organisms is None:
        target_organisms = {
            9606: 'Homo sapiens',
            10090: 'Mus musculus'
        }
    
    # 타겟 Tax ID 리스트 (문자열로 변환하여 비교 준비)
    target_tax_ids = [str(tid) for tid in target_organisms.keys()]
    
    URL = "https://ftp.ncbi.nih.gov/gene/DATA/gene_info.gz"
    
    # 출력 파일명 정의 (통합 파일)
    FILE_MAIN = "ncbi_genes.csv"
    FILE_SYNONYMS = "ncbi_synonyms.csv"
    FILE_ENSEMBL = "ncbi_ensembl.csv"
    
    print(f"\n{'='*70}")
    print(f"NCBI Gene Info 통합 처리 시작")
    print(f"{'='*70}")
    print(f"대상 종: {target_organisms}")
    print(f"URL: {URL}")
    print("\n[출력 파일]")
    print(f"1. 메인 정보: {FILE_MAIN}")
    print(f"2. Synonyms : {FILE_SYNONYMS}")
    print(f"3. Ensembl  : {FILE_ENSEMBL}\n")
    
    # 데이터 읽기 (Chunk 방식)
    chunk_iter = pd.read_csv(
        URL, 
        sep='\t', 
        compression='gzip', 
        dtype=str,
        chunksize=200000 
    )
    
    filtered_chunks = []
    processed_rows = 0
    
    try:
        print("데이터 스캔 및 필터링 중...")
        for chunk in chunk_iter:
            tax_col = chunk.columns[0] # '#tax_id' 컬럼
            
            # 타겟 Tax ID에 해당하는 행만 필터링
            temp_df = chunk[chunk[tax_col].isin(target_tax_ids)]
            
            if not temp_df.empty:
                filtered_chunks.append(temp_df)
            
            processed_rows += len(chunk)
            sys.stdout.write(f"\r처리된 행 수: {processed_rows:,}")
            sys.stdout.flush()
        
        print("\n\n데이터 병합 및 정규화 작업 시작...")
        
        if not filtered_chunks:
            print(f"\n⚠️ 지정된 TaxID에 대한 데이터를 찾지 못했습니다.")
            return None, None, None

        full_df = pd.concat(filtered_chunks, ignore_index=True)
        
        # ==========================================
        # 0. 공통 전처리
        # ==========================================
        # 필요한 원본 컬럼만 선택 (#tax_id 포함)
        cols_needed = [full_df.columns[0], 'GeneID', 'Symbol', 'description', 'type_of_gene', 'Synonyms', 'dbXrefs']
        full_df = full_df[cols_needed].copy()
        
        rename_map = {
            full_df.columns[0]: 'tax_id', # #tax_id -> tax_id
            'GeneID': 'entrez_id',
            'Symbol': 'gene_symbol',
            'description': 'description',
            'type_of_gene': 'gene_type',
            'Synonyms': 'synonyms_raw', 
            'dbXrefs': 'db_xrefs'
        }
        full_df.rename(columns=rename_map, inplace=True)
        full_df.replace('-', np.nan, inplace=True)

        # [핵심] Organism 이름 매핑 추가
        # tax_id 컬럼을 int로 변환 후 매핑 (안전하게)
        full_df['tax_id'] = full_df['tax_id'].astype(int)
        full_df['organism'] = full_df['tax_id'].map(target_organisms)

        # ==========================================
        # 1. FILE 1: Main Info Table
        # (entrez_id, tax_id, organism, gene_symbol...)
        # ==========================================
        df_main = full_df[[
            'entrez_id', 'tax_id', 'organism', 'gene_symbol', 'description', 'gene_type'
        ]].drop_duplicates()
        
        # ==========================================
        # 2. FILE 2: Synonyms Table
        # ==========================================
        print("Synonyms 분리 중...")
        df_syn_temp = full_df[['entrez_id', 'tax_id', 'organism', 'synonyms_raw']].dropna()
        df_syn_temp['synonym'] = df_syn_temp['synonyms_raw'].str.split('|')
        
        df_synonyms = df_syn_temp.explode('synonym')[[
            'entrez_id', 'tax_id', 'organism', 'synonym'
        ]]
        
        df_synonyms['synonym'] = df_synonyms['synonym'].str.strip()
        df_synonyms = df_synonyms[df_synonyms['synonym'] != '']
        df_synonyms = df_synonyms.drop_duplicates()
        
        # ==========================================
        # 3. FILE 3: Ensembl Map Table
        # ==========================================
        print("Ensembl ID 추출 중...")
        df_ensembl_temp = full_df[['entrez_id', 'tax_id', 'organism', 'db_xrefs']].dropna()
        
        # 정규표현식으로 Ensembl ID 추출
        df_ensembl_temp['ensembl_id'] = df_ensembl_temp['db_xrefs'].str.extract(r'Ensembl:(ENS[A-Z0-9]+)')
        
        df_ensembl = df_ensembl_temp[[
            'entrez_id', 'tax_id', 'organism', 'ensembl_id'
        ]].dropna().drop_duplicates()

        # ==========================================
        # 4. 결과 저장
        # ==========================================
        os.makedirs("ncbi_data", exist_ok=True) # 폴더 생성
        
        path_main = os.path.join("ncbi-data", FILE_MAIN)
        path_syn = os.path.join("ncbi-data", FILE_SYNONYMS)
        path_ens = os.path.join("ncbi-data", FILE_ENSEMBL)

        df_main.to_csv(path_main, index=False)
        df_synonyms.to_csv(path_syn, index=False)
        df_ensembl.to_csv(path_ens, index=False)
        
        print(f"\n{'='*70}")
        print(f"처리 완료 및 저장 성공")
        print(f"{'='*70}")
        
        print(f"1. [Main]    {path_main} ({len(df_main):,} rows)")
        print(f"2. [Synonym] {path_syn} ({len(df_synonyms):,} rows)")
        print(f"3. [Ensembl] {path_ens} ({len(df_ensembl):,} rows)")
        
        # 데이터 분포 출력
        print("\n[데이터 분포 - Main Table]")
        print(df_main['organism'].value_counts())

        return df_main, df_synonyms, df_ensembl
            
    except Exception as e:
        print(f"\n에러 발생: {e}")
        import traceback
        traceback.print_exc()
        return None, None, None

# ==========================================
# 실행부
# ==========================================
if __name__ == "__main__":
    # 원하는 종들을 딕셔너리로 정의 (TaxID : 이름)
    targets = {
        9606: 'Homo sapiens',   # Human
        10090: 'Mus musculus'   # Mouse
    }
    
    # 함수 실행
    df_main, df_syn, df_ens = process_ncbi_gene_info_integrated(targets)
    
    if df_main is not None:
        print("\n--- Main Table Preview ---")
        print(df_main.head())
        print(df_main.tail())


NCBI Gene Info 통합 처리 시작
대상 종: {9606: 'Homo sapiens', 10090: 'Mus musculus'}
URL: https://ftp.ncbi.nih.gov/gene/DATA/gene_info.gz

[출력 파일]
1. 메인 정보: ncbi_genes_integrated.csv
2. Synonyms : ncbi_synonyms_integrated.csv
3. Ensembl  : ncbi_ensembl_integrated.csv

데이터 스캔 및 필터링 중...
처리된 행 수: 64,761,031

데이터 병합 및 정규화 작업 시작...
Synonyms 분리 중...
Ensembl ID 추출 중...

처리 완료 및 저장 성공
1. [Main]    ncbi_data/ncbi_genes_integrated.csv (305,973 rows)
2. [Synonym] ncbi_data/ncbi_synonyms_integrated.csv (150,873 rows)
3. [Ensembl] ncbi_data/ncbi_ensembl_integrated.csv (73,400 rows)

[데이터 분포 - Main Table]
organism
Homo sapiens    193718
Mus musculus    112255
Name: count, dtype: int64

--- Main Table Preview ---
  entrez_id  tax_id      organism gene_symbol                     description  \
0         1    9606  Homo sapiens        A1BG          alpha-1-B glycoprotein   
1         2    9606  Homo sapiens         A2M           alpha-2-macroglobulin   
2         9    9606  Homo sapiens        NAT1           